# Draft flows — derived bundles, adjustments, Euler

This notebook is the sequel to `01-flows.ipynb`. Same SIR × age × 3×3 location
map, same three flow kinds, same rule that a **flow** is one named object that
actualizes to many edges. Nothing here is the public `summer4` API
(`explorations/flows/` is out of package).

`01` stopped at a single `vf(t, y, params)` evaluation. Here we ask three
follow-up questions, and we **plot** the answers so the numbers have a shape:

1. Can a **single derived proxy** carry a time-varying migration matrix — an
   initial data-derived baseline plus a dynamic seasonal factor?
2. How do we write **adjustments** (Multiply by default, Overwrite, Transform)
   that see the previous aligned rate and can mention other `FlowRef`s?
3. Does a **minimal Euler** stepper, itself wrapped in `jax.jit`, run this
   end-to-end?

Plots use **Polars** frames and **Plotly**. They are ordinary Python objects
(no IPython magics); in a Jupyter kernel they render as Plotly figures.


## 1. The compartment table (same map as `01`)

A `Property` is a named set of mutually exclusive traits. A `PropertyMap` is
the ragged table of compartments those properties create. Each **row** is one
compartment; `select` returns its integer indices.

We keep the `01` taxonomy so the sequel is comparable: SIR, fully stratified
by age and a **3×3 location grid**, with severity only on `I`. Location traits
are `r{row}c{col}`.


In [ ]:
from typing import NamedTuple

import numpy as np
import plotly.express as px
import polars as pl

from explorations.flows.prototype import (
    EntryFlow,
    ExitFlow,
    FieldRef,
    FlowModel,
    Overwrite,
    TraitChain,
    TraitMatrix,
    Transform,
    TransitionFlow,
    derived_refs,
    euler,
)
from summer4 import Everything, Property, PropertyMap

GRID = 3
CELLS = tuple(f"r{row}c{col}" for row in range(GRID) for col in range(GRID))
assert CELLS == ("r0c0", "r0c1", "r0c2", "r1c0", "r1c1", "r1c2", "r2c0", "r2c1", "r2c2")

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
location = Property("location", CELLS)
severity = Property("severity", ("mild", "severe"))

pm = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(location)
    .stratify(severity, where=state["I"])
)
# S×3×9 + I×3×9×2 + R×3×9 = 27 + 54 + 27 = 108
assert pm.size == 108
assert pm.select(state["S"]).size == 27
assert pm.select(state["I"]).size == 54
pm.size


`PropertyMap.select(selector)` is still the only query the join needs. A flow
stores selectors; it does **not** expand into one Python object per edge.

What *is* new is how much structure we can hang off a `FieldRef`, and that we
can step the compiled field in time.


## 2. Nested `derived_refs` — one proxy for a bundle

In `01`, `derived_refs(Derived)` built a flat bag of `FieldRef`s (`D.foi`,
`D.death_rate`). Tab-complete only offered real schema fields; at vector-field
time the compiled function walked those paths on the `NamedTuple` returned by
`compute_derived_params`.

Migration is not a scalar. It has a **baseline dest×source matrix** (from
data, or from a neighbour construction) and a **seasonal multiplier** that
depends on `t`. We want to pass that as **one derived argument**, not two
unrelated names the reader has to keep in their head.

`derived_refs` now recurses: a field whose annotation is itself a NamedTuple
becomes a nested instance of that schema, filled with prefixed `FieldRef`s.
`D.migration.baseline` is `FieldRef(("migration", "baseline"))`. A flat
`migration_rates` field is the already-combined matrix — same compactness,
no unpacking.

We do **not** auto-multiply a looked-up NamedTuple. That would be too
magical. The “single proxy” is either the combined field or the nested object
you pass into ordinary `*`.


In [ ]:
class Migration(NamedTuple):
    baseline: object  # dest × source ndarray at runtime
    seasonal: float


class Derived(NamedTuple):
    foi: float
    death_rate: float
    foi_cap: float
    seasonal: float
    migration: Migration
    migration_rates: object


D = derived_refs(Derived)
assert D.foi == FieldRef(("foi",))
assert D.migration.baseline == FieldRef(("migration", "baseline"))
assert D.migration.seasonal == FieldRef(("migration", "seasonal"))
assert D.migration_rates == FieldRef(("migration_rates",))
D.migration


## 3. Migration topology stays static; rates can move

`TraitMatrix(location, M)` is still the pairing for a (possibly sparse)
transition matrix. Convention: **`M` is dest × source**. Entry `M[j, i]` is
the per-capita rate from location trait `i` to trait `j`. Zero entries are
not edges.

In `01` the hop *values* lived in that matrix and were baked onto each edge's
`scale` at actualize time. That is fine for a constant rate. It is **not**
fine if the rates should vary with `t`: actualize runs once, and changing
which pairs exist would rebuild index arrays and break `jit`.

So we split the job:

- `TraitMatrix` holds a **0/1 neighbour mask** (topology only).
- The flow `rate` is a dest×source array from derived params, gathered onto
  those edges each step as `rate[dest_code, src_code]`.

We still want movement only between **immediate 4-neighbours** (no diagonals,
no wrap-around):

```
r0c0 — r0c1 — r0c2
  |      |      |
r1c0 — r1c1 — r1c2
  |      |      |
r2c0 — r2c1 — r2c2
```

A 3×3 grid has 12 undirected adjacencies and therefore **24 directed**
location edges. Leftover state / age / severity still match, so an infectious
mild child in `r1c1` only moves to the same compartment in a neighbouring
cell.


In [ ]:
def parse_cell(name: str) -> tuple[int, int]:
    """Decode ``r{row}c{col}`` into integer grid coordinates."""
    return int(name[1]), int(name[3])


def neighbor_mask(cells: tuple[str, ...]) -> np.ndarray:
    """Dest×source 0/1 mask on 4-neighbour pairs."""
    index = {name: i for i, name in enumerate(cells)}
    matrix = np.zeros((len(cells), len(cells)))
    for src in cells:
        row, col = parse_cell(src)
        for d_row, d_col in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            dest = f"r{row + d_row}c{col + d_col}"
            if dest in index:
                matrix[index[dest], index[src]] = 1.0
    return matrix


BASE_HOP = 0.05
AMP = 0.5
OMEGA = 1.0
mask = neighbor_mask(CELLS)
baseline = mask * BASE_HOP
assert mask.shape == (9, 9)
assert int(np.count_nonzero(mask)) == 24
assert mask[CELLS.index("r0c1"), CELLS.index("r0c0")] == 1.0  # east of NW
assert mask[CELLS.index("r0c2"), CELLS.index("r0c0")] == 0.0  # not a neighbour
assert mask[CELLS.index("r1c1"), CELLS.index("r0c0")] == 0.0  # diagonal
np.testing.assert_allclose(mask, mask.T)
int(np.count_nonzero(mask))


The heatmap is dest × source: a lit cell means “this destination accepts
hoppers from that source”. The diagonal is empty (no self-loops). Corner
cells have two neighbours; the centre has four.


In [ ]:
mig_long = pl.DataFrame(
    [
        {"source": src, "dest": dest, "hop_rate": float(baseline[j, i])}
        for i, src in enumerate(CELLS)
        for j, dest in enumerate(CELLS)
    ]
)
assert mig_long.filter(pl.col("hop_rate") > 0).height == 24
px.imshow(
    baseline,
    x=list(CELLS),
    y=list(CELLS),
    labels={"x": "source", "y": "dest", "color": "hop_rate"},
    title="Baseline dest × source hop rates (0/1 mask × 0.05)",
    zmin=0.0,
    zmax=BASE_HOP,
).update_layout(width=520, height=440, yaxis={"autorange": "reversed"})


## 4. Time-varying rates from one `FieldRef`

`compile(derived_fn=...)` calls `derived_fn(params, y=y, t=t)` every step.
That is how `01` built a frequency-dependent FOI without an
`InfectionFrequencyFlow` class. The same hook can rebuild a **matrix**.

Here the seasonal factor is `1 + amp * sin(omega * t)`. At `t = 0` it is `1`
(baseline only). At `t = π / (2 ω)` it is `1 + amp`.
`compute_derived_params` writes both the nested `Migration` bundle and the
already-multiplied `migration_rates` array so the next two sections can
compare those styles.

`make_derived_fn(xp)` takes `numpy` or `jax.numpy` so the same formula works
inside a jitted stepper.


In [ ]:
I_idx = pm.select(state["I"])


def make_derived_fn(xp):
    def compute_derived_params(params, y=None, t=None):
        t_v = 0.0 if t is None else t
        seasonal = 1.0 + params["amp"] * xp.sin(params["omega"] * t_v)
        infected = xp.sum(y[I_idx]) if y is not None else 0.0
        population = xp.sum(y) if y is not None else 1.0
        foi = params["contact"] * infected / population
        return Derived(
            foi=foi,
            death_rate=params["death_rate"],
            foi_cap=params["foi_cap"],
            seasonal=seasonal,
            migration=Migration(baseline=xp.asarray(baseline), seasonal=seasonal),
            migration_rates=xp.asarray(baseline) * seasonal,
        )

    return compute_derived_params


params = {
    "contact": 0.4,
    "death_rate": 0.01,
    "foi_cap": 0.02,
    "amp": AMP,
    "omega": OMEGA,
}

y = np.zeros(pm.size)
for i, cell in enumerate(CELLS):
    y[pm.select(location[cell] & state["S"])] = 50.0 * (i + 1)
y[pm.select(state["I"] & severity["mild"])] = 4.0
y[pm.select(state["I"] & severity["severe"])] = 1.0

t_grid = np.linspace(0.0, 2.0 * np.pi / OMEGA, 80)
seasonal_df = pl.DataFrame(
    {
        "t": t_grid,
        "seasonal": 1.0 + AMP * np.sin(OMEGA * t_grid),
    }
)
px.line(
    seasonal_df,
    x="t",
    y="seasonal",
    title="Seasonal multiplier  1 + amp·sin(ω t)",
).update_layout(width=640, height=280)


Seed populations increase along the grid (`r0c0` light, `r2c2` heavy) so
migration is visible as a downhill flow. The migration-only field at two
times, **same `y`**, must scale by the seasonal factor and conserve mass.


In [ ]:
def location_mass(vec: np.ndarray) -> pl.DataFrame:
    """Sum ``vec`` inside each location trait."""
    return pl.DataFrame(
        {
            "location": list(CELLS),
            "mass": [float(vec[pm.select(location[cell])].sum()) for cell in CELLS],
        }
    )


mig_model = FlowModel(pm)
mig_model.add_flow(
    TransitionFlow(
        "migration",
        location.present(),
        location.present(),
        D.migration_rates,
        pairing=TraitMatrix(location, mask),
    )
)
vf_mig = mig_model.compile(derived_fn=make_derived_fn(np))

dy0 = vf_mig(0.0, y, params)
t_peak = float(np.pi / (2.0 * OMEGA))
dy_peak = vf_mig(t_peak, y, params)
np.testing.assert_allclose(dy0.sum(), 0.0, atol=1e-12)
np.testing.assert_allclose(dy_peak.sum(), 0.0, atol=1e-12)
np.testing.assert_allclose(dy_peak, dy0 * (1.0 + AMP), rtol=1e-8, atol=1e-12)

loc_y = location_mass(y).rename({"mass": "population"})
px.bar(
    loc_y,
    x="location",
    y="population",
    title="Initial population by location (S seeded 50, 100, … along the grid)",
).update_layout(width=640, height=280)


`dy` at the seasonal peak is the `t = 0` field scaled by `1 + amp`. Corners
with more people lose more; the centre is closer to balanced. Totals stay
zero — this is a closed transition.


In [ ]:
dy_by_loc = location_mass(dy0).rename({"mass": "dy_t0"}).join(
    location_mass(dy_peak).rename({"mass": "dy_peak"}),
    on="location",
)
dy_long = dy_by_loc.unpivot(
    index="location",
    on=["dy_t0", "dy_peak"],
    variable_name="when",
    value_name="dy",
)
px.bar(
    dy_long,
    x="location",
    y="dy",
    color="when",
    barmode="group",
    title="Migration-only dy by location  (peak = t0 × 1.5)",
).update_layout(width=640, height=320, xaxis_tickangle=45)


## 5. The same rate as `D.migration.baseline * D.migration.seasonal`

The nested proxy is one argument you pass into the existing rate algebra.
`BinOp` already knows `*`. The dest×source result aligns the same way as the
combined field — so the two styles are interchangeable, and you pick whichever
reads better at the call site.


In [ ]:
nested_model = FlowModel(pm)
nested_model.add_flow(
    TransitionFlow(
        "migration",
        location.present(),
        location.present(),
        D.migration.baseline * D.migration.seasonal,
        pairing=TraitMatrix(location, mask),
    )
)
dy_nested = nested_model.compile(derived_fn=make_derived_fn(np))(t_peak, y, params)
np.testing.assert_allclose(dy_nested, dy_peak, rtol=1e-8, atol=1e-12)
dy_nested.sum()


## 6. Adjustments — Multiply, Overwrite, Transform

`01` treated stratum-specific rates as “just use a `PropertyData`”. That still
works, but it is not how summer2 authors thought about *interventions*: a
base parameter, then a short list of modifications.

`adjust=` is a **pipeline on the already-aligned per-edge rate** (after
pairing `scale`, before `* y[src]`). Each step’s input is the previous step’s
output. Extra arguments are the same expression language as rates
(`FieldRef`, `FlowRef`, `BinOp`).

| Kind | Meaning |
| --- | --- |
| bare scalar / `RateOps` | **Multiply** (summer2 `enforce_multiply`) |
| `Overwrite(value, where=None)` | replace `prev` with `value` |
| `Transform(fn, *args, where=None)` | `fn(prev, *resolved_args)` |

`where=` is a selector on the gather index (`src` for transition/exit, `dest`
for entry). Unmatched edges keep `prev`.

`fn` must be vectorized over edges (`np.minimum`, `jnp.minimum`). A Python
`if` over edges will not `jit`.

`D.foi * D.seasonal` on the **rate** is still the way to bake a multiply into
the definition. Use `adjust=` when you need **order**, a **previous value**,
or a **stratum mask**.


In [ ]:
inf_only = FlowModel(pm)
inf_only.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        D.foi,
        split={severity: {"mild": 0.8, "severe": 0.2}},
        adjust=[
            D.seasonal,  # Multiply
            Overwrite(0.0, where=age["0-4"]),
            Transform(np.minimum, D.foi_cap),
        ],
    )
)
vf_inf = inf_only.compile(derived_fn=make_derived_fn(np))
dy_inf = vf_inf(0.0, y, params)
np.testing.assert_allclose(dy_inf[pm.select(state["S"] & age["0-4"])], 0.0)
np.testing.assert_allclose(dy_inf.sum(), 0.0, atol=1e-12)

derived0 = make_derived_fn(np)(params, y=y, t=0.0)
older_s = pm.select(state["S"] & ~age["0-4"])
rate = min(float(derived0.foi) * float(derived0.seasonal), float(derived0.foi_cap))
np.testing.assert_allclose(-dy_inf[older_s], rate * y[older_s])

ages = ("0-4", "5-9", "10+")
age_out = pl.DataFrame(
    {
        "age": list(ages),
        "S_outflow": [
            -float(dy_inf[pm.select(state["S"] & age[band])].sum()) for band in ages
        ],
    }
)
assert float(age_out.filter(pl.col("age") == "0-4")["S_outflow"][0]) == 0.0
px.bar(
    age_out,
    x="age",
    y="S_outflow",
    title="Infection-only S outflow by age  (0–4 overwritten to rate 0)",
).update_layout(width=480, height=280)


`Transform` can also consume **another flow’s already-computed mass**.
`add_flow` returns a `FlowRef`; topo order evaluates `death` before
`infection`, so `death.sum()` is a scalar argument to `fn(prev, deaths)`.
Cycles still raise.


In [ ]:
death_first = FlowModel(pm)
death_ref = death_first.add_flow(ExitFlow("death", Everything(), D.death_rate))
death_first.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        D.foi,
        split={severity: {"mild": 0.8, "severe": 0.2}},
        adjust=[Transform(lambda prev, deaths: prev / (1.0 + deaths), death_ref.sum())],
    )
)
dy_combo = death_first.compile(derived_fn=make_derived_fn(np))(0.0, y, params)
death_sum = params["death_rate"] * float(y.sum())
inf_rate = float(derived0.foi) / (1.0 + death_sum)
s_idx = pm.select(state["S"])
np.testing.assert_allclose(
    -dy_combo[s_idx],
    params["death_rate"] * y[s_idx] + inf_rate * y[s_idx],
)
float(death_sum)


## 7. Euler — a few steps, including inside `jax.jit`

The first spike stopped at `vf`. Time-varying matrices and `adjust=` are only
convincing if they survive **several steps**, including as the body of
`lax.scan` under `jax.jit`.

`euler(vf, t0, y0, params, dt=..., steps=...)` is forward Euler:
`y ← y + dt * vf(t, y, params)`. NumPy uses a Python loop; JAX uses
`lax.scan` so the **stepper** is the jit target, not an unrolled loop.

The model below is the `01` story plus the new knobs: infection gets a
seasonal Multiply and a young Overwrite; migration uses the nested proxy
times the neighbour mask; death still replaces into young-S per location.
Replacement + closed transitions imply `sum(y)` is constant.

A frozen-rate control (`amp = 0`) must finish somewhere else — that is how
we know `t` actually entered `compute_derived_params`.


In [ ]:
import jax
import jax.numpy as jnp

model = FlowModel(pm)
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        D.foi,
        split={severity: {"mild": 0.8, "severe": 0.2}},
        adjust=[D.seasonal, Overwrite(0.0, where=age["0-4"])],
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
model.add_flow(
    TransitionFlow(
        "ageing",
        age.present(),
        age.present(),
        1.0,
        pairing=TraitChain(
            age,
            (("0-4", "5-9"), ("5-9", "10+")),
            rates=(1.0 / 5.0, 1.0 / 5.0),
        ),
    )
)
model.add_flow(
    TransitionFlow(
        "migration",
        location.present(),
        location.present(),
        D.migration.baseline * D.migration.seasonal,
        pairing=TraitMatrix(location, mask),
    )
)
death = model.add_flow(ExitFlow("death", Everything(), D.death_rate))
model.add_flow(EntryFlow("birth", age["0-4"] & state["S"], death.sum_over(location)))

DT = 0.25
STEPS = 8
vf_np = model.compile(derived_fn=make_derived_fn(np))
y_np = euler(vf_np, 0.0, y, params, dt=DT, steps=STEPS)
np.testing.assert_allclose(y_np.sum(), y.sum(), atol=1e-10)

vf_jax = model.compile(derived_fn=make_derived_fn(jnp), backend="jax")
step = jax.jit(lambda state: euler(vf_jax, 0.0, state, params, dt=DT, steps=STEPS))
y_j = np.asarray(step(jnp.asarray(y)))
np.testing.assert_allclose(y_j, y_np, rtol=1e-5)

params_const = dict(params)
params_const["amp"] = 0.0
y_const = euler(vf_np, 0.0, y, params_const, dt=DT, steps=STEPS)
assert not np.allclose(y_np, y_const)
float(y_np.sum())


`euler` returns only the final state (no timeseries type). For a picture we
record a longer NumPy walk — one period of the sinusoid — under both the
seasonal params and the frozen control. SIR totals stay a closed system;
location `S` shows the grid relaxing downhill, faster when the seasonal
multiplier is above 1.


In [ ]:
def record_sir(
    vf, y0: np.ndarray, run_params: dict, dt: float, steps: int
) -> pl.DataFrame:
    """Walk Euler on ``vf`` and return SIR totals vs time."""
    y_t = np.array(y0, copy=True)
    t = 0.0
    rows: list[dict[str, float]] = []
    for _ in range(steps + 1):
        rows.append(
            {
                "t": t,
                "S": float(y_t[pm.select(state["S"])].sum()),
                "I": float(y_t[pm.select(state["I"])].sum()),
                "R": float(y_t[pm.select(state["R"])].sum()),
            }
        )
        y_t = y_t + dt * vf(t, y_t, run_params)
        t = t + dt
    return pl.DataFrame(rows)


PLOT_STEPS = 512
sir_seasonal = record_sir(vf_np, y, params, DT, PLOT_STEPS).with_columns(
    pl.lit("seasonal").alias("run")
)
sir_frozen = record_sir(vf_np, y, params_const, DT, PLOT_STEPS).with_columns(
    pl.lit("frozen").alias("run")
)
sir_long = (
    pl.concat([sir_seasonal, sir_frozen])
    .unpivot(
        index=["t", "run"],
        on=["S", "I", "R"],
        variable_name="compartment",
        value_name="population",
    )
    .with_columns((pl.col("compartment") + " (" + pl.col("run") + ")").alias("series"))
)
px.line(
    sir_long,
    x="t",
    y="population",
    color="series",
    title="Euler walk — SIR totals (seasonal vs frozen migration/FOI)",
).update_layout(width=1000, height=600)


The same walk, coloured by location, is the migration story: heavy cells lose
`S`, light cells gain, and the seasonal run pulls ahead of the frozen run
whenever `sin(ω t) > 0`.


In [ ]:
def record_location_s(
    vf, y0: np.ndarray, run_params: dict, dt: float, steps: int
) -> pl.DataFrame:
    """Walk Euler and return S mass in each location vs time."""
    y_t = np.array(y0, copy=True)
    t = 0.0
    rows: list[dict[str, float | str]] = []
    for _ in range(steps + 1):
        for cell in CELLS:
            rows.append(
                {
                    "t": t,
                    "location": cell,
                    "S": float(y_t[pm.select(location[cell] & state["S"])].sum()),
                }
            )
        y_t = y_t + dt * vf(t, y_t, run_params)
        t = t + dt
    return pl.DataFrame(rows)


loc_s = record_location_s(vf_np, y, params, DT, PLOT_STEPS)
px.line(
    loc_s,
    x="t",
    y="S",
    color="location",
    title="S by location under seasonal migration",
).update_layout(width=1000, height=320)
